# Name That LLM: which LLM layer knows the author? (sweep + blend A/B)

**Goal:** find the language-model layer whose hidden states best identify the 18 authors,
then test whether blending that "LM model" with our TF-IDF champion (`lr`, public LB 0.320)
is genuinely better.

Part of the learning-in-public MLOps project
[github.com/Ryqn0/name-that-llm](https://github.com/Ryqn0/name-that-llm), plan P3b.
This CPU notebook reads the features that the GPU notebook
[name-that-llm-lm-features](https://www.kaggle.com/code/ryqn00/name-that-llm-lm-features)
extracted. Chaining notebooks spends GPU time only where a GPU is needed.

**How we avoid fooling ourselves:** every choice (model size, layer, regularization, blend
weight) is made on cross-validation folds with **seed 7**. The final numbers are then
reported on **different** folds (seeds 42, 43, 44). Choosing the best of many options on the
same folds you report on inflates the score, because part of the "best" is just luck.

In [ ]:
%pip install -q "name-that-llm[track] @ git+https://github.com/Ryqn0/name-that-llm@v0.5.0"

In [ ]:
import glob
import os
import time

import numpy as np
import pandas as pd
import yaml

from name_that_llm.blend import best_blend_weight, blend
from name_that_llm.config import BlendConfig, load_config
from name_that_llm.data import LABELS, load_train
from name_that_llm.lm import select_layers
from name_that_llm.metrics import prompt_bootstrap_ci, score
from name_that_llm.train import run_cv, run_cv_features

# Tracking is optional: without the Kaggle Secret, everything still runs.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    TRACK = True
except Exception as exc:
    TRACK = False
    print(f"W&B tracking is OFF ({type(exc).__name__}). Results are still computed.")

train = load_train()

# Row alignment (plan Review Focus 1): the features are plain arrays matched to labels by
# POSITION. The GPU notebook saved its row order; any mismatch would make every number here
# meaningless, so we stop immediately rather than compute garbage.
ids_files = glob.glob("/kaggle/input/**/train_ids.npy", recursive=True)
assert len(ids_files) == 1, f"expected one train_ids.npy, found {ids_files}"
assert (np.load(ids_files[0], allow_pickle=True) == train["id"].to_numpy()).all(), (
    "row order differs!"
)

FEATURES = {}
for path in sorted(glob.glob("/kaggle/input/**/features-*.npy", recursive=True)):
    variation = os.path.basename(path).removeprefix("features-").removesuffix(".npy")
    FEATURES[variation] = np.load(path)
    print(f"{variation:10s} {FEATURES[variation].shape} {FEATURES[variation].dtype}")
assert FEATURES, "no features found: is the lm-features notebook attached?"

## 1. Sweep the layers (choices on seed-7 folds)

For each model size we cross-validate a logistic regression on **every 2nd layer**, then on
both neighbors of that size's best layer. Each point is a full 5-fold, prompt-grouped CV, so
it's an honest estimate of how the layer would do on unseen prompts.

In [ ]:
SWEEP_SEED = 7
y = train["model"].to_numpy()
groups = train["prompt_id"].to_numpy()


def cv_lm(variation, layer, window=1, C=1.0, seed=SWEEP_SEED):
    return run_cv_features(select_layers(FEATURES[variation], layer, window), train, C=C, seed=seed)


rows = []
for variation, F in FEATURES.items():
    start, tried = time.time(), {}
    for layer in range(0, len(F), 2):
        tried[layer] = cv_lm(variation, layer).score
    best = max(tried, key=tried.get)
    for layer in (best - 1, best + 1):
        if 0 <= layer < len(F) and layer not in tried:
            tried[layer] = cv_lm(variation, layer).score
    for layer, s in sorted(tried.items()):
        rows.append(
            {
                "variation": variation,
                "layer": layer,
                "n_layers": len(F),
                "relative_depth": layer / (len(F) - 1),
                "cv": s,
            }
        )
    best = max(tried, key=tried.get)
    print(
        f"{variation:10s} best layer {best:2d}/{len(F) - 1}: CV {tried[best]:.4f} "
        f"({(time.time() - start) / 60:.0f} min)",
        flush=True,
    )

sweep = pd.DataFrame(rows)
lr_cfg = load_config("lr")
lr_sweep = run_cv(lr_cfg.model_copy(update={"seed": SWEEP_SEED}), train)
print(f"TF-IDF lr on the same seed-7 folds: {lr_sweep.score:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Colors: the first three slots of a colorblind-checked palette (validated all-pairs). The
# aqua slot is below 3:1 contrast on the light surface, so every line also gets a direct text
# label, and the full numbers are in the table below.
SURFACE, INK, INK_2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]

fig, ax = plt.subplots(figsize=(9, 4.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
# strict=False: there may be fewer models than colors (a model can be skipped in extraction).
for color, (variation, part) in zip(COLORS, sweep.groupby("variation", sort=False), strict=False):
    ax.plot(
        part["relative_depth"],
        part["cv"],
        color=color,
        linewidth=2,
        marker="o",
        markersize=6,
        label=f"Qwen 3 {variation}",
    )
    last = part.iloc[-1]
    ax.annotate(
        variation,
        (last["relative_depth"], last["cv"]),
        xytext=(6, 0),
        textcoords="offset points",
        va="center",
        color=INK_2,
        fontsize=9,
    )
ax.axhline(lr_sweep.score, color=MUTED, linestyle="--", linewidth=1.5)
ax.annotate(
    "TF-IDF lr (same folds)",
    (0.0, lr_sweep.score),
    xytext=(2, 4),
    textcoords="offset points",
    color=MUTED,
    fontsize=9,
)
ax.set_xlabel("relative depth (0 = embeddings, 1 = last layer)", color=INK_2)
ax.set_ylabel("CV balanced accuracy (seed-7 folds)", color=INK_2)
ax.set_title("Which layer identifies the author best?", loc="left", fontweight="bold", color=INK)
ax.grid(color=GRID, linewidth=0.6)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3)
plt.show()

sweep.pivot(index="layer", columns="variation", values="cv").round(4)  # the table view

## 2. Refine around the best layer

Around the overall best (model size, layer): average a **window** of 1, 2 or 4 neighboring
layers (it smooths layer-to-layer noise) and try 4 regularization strengths `C`. A smaller
`C` means a simpler model, which can generalize better with only 150 prompts.

In [ ]:
best_row = sweep.loc[sweep["cv"].idxmax()]
VARIATION, BEST_LAYER = best_row["variation"], int(best_row["layer"])
n_layers = len(FEATURES[VARIATION])

refine = []
for window in (1, 2, 4):
    # Center the window on the best layer, kept inside the valid range.
    start = min(max(BEST_LAYER - window // 2, 0), n_layers - window)
    for C in (0.1, 0.3, 1.0, 3.0):
        refine.append(
            {
                "layer": start,
                "window": window,
                "C": C,
                "cv": cv_lm(VARIATION, start, window, C).score,
            }
        )
refine = pd.DataFrame(refine).sort_values("cv", ascending=False)
best = refine.iloc[0]
LAYER, WINDOW, C = int(best["layer"]), int(best["window"]), float(best["C"])
print(f"best: {VARIATION}, layers {LAYER}..{LAYER + WINDOW - 1}, C={C}: CV {best['cv']:.4f}")
refine.round(4).head(6)

## 3. Blend weight, then the honest A/B test on fresh folds

The blend weight is chosen on the seed-7 folds too. Then **everything is re-run on seeds 42,
43 and 44**: the LM model alone, the TF-IDF `lr`, and their blend with the chosen weight.
The prompt-bootstrap on seed 42 decides: promote the blend only if the whole 95% interval of
"blend minus lr" is above 0.

In [ ]:
lm_sweep = cv_lm(VARIATION, LAYER, WINDOW, C)
WEIGHT, blend_sweep_score = best_blend_weight(lr_sweep.oof, lm_sweep.oof, y)
print(f"blend weight on the LM side: {WEIGHT} (seed-7 CV {blend_sweep_score:.4f})")

labels = np.asarray(LABELS)
report, kept = [], {}
for seed in (42, 43, 44):
    text = run_cv(lr_cfg.model_copy(update={"seed": seed}), train)
    lm = cv_lm(VARIATION, LAYER, WINDOW, C, seed=seed)
    mixed = labels[blend([text.oof, lm.oof], [1 - WEIGHT, WEIGHT]).argmax(axis=1)]
    report.append({"seed": seed, "lr": text.score, "lm": lm.score, "blend": score(y, mixed)})
    kept[seed] = (text.pred, mixed)
report = pd.DataFrame(report).set_index("seed")
report.loc["mean"] = report.mean()

diff, lo, hi = prompt_bootstrap_ci(y, kept[42][0], kept[42][1], groups, n_boot=2000, seed=0)
PROMOTE = lo > 0
print(
    f"blend minus lr (seed 42): {diff:+.4f}, 95% CI [{lo:+.4f}, {hi:+.4f}] -> "
    + ("PROMOTE the blend" if PROMOTE else "keep lr: the gain isn't clear")
)
report.round(4)

In [ ]:
cfg = BlendConfig(
    name="lm_blend",
    text_config="lr",
    lm_variation=VARIATION,
    layer=LAYER,
    layer_window=WINDOW,
    C=C,
    weight=WEIGHT,
)
with open("/kaggle/working/lm_blend.yaml", "w") as f:
    yaml.safe_dump(cfg.model_dump(), f, sort_keys=False)
sweep.to_csv("/kaggle/working/lm_sweep.csv", index=False)
print(yaml.safe_dump(cfg.model_dump(), sort_keys=False))

if TRACK:
    import wandb

    run = wandb.init(
        project="name-that-llm", job_type="lm-sweep", name="lm-sweep", config=cfg.model_dump()
    )
    run.log({"sweep": wandb.Table(dataframe=sweep)})
    run.summary.update(
        {
            "cv_lr_mean": report.loc["mean", "lr"],
            "cv_lm_mean": report.loc["mean", "lm"],
            "cv_blend_mean": report.loc["mean", "blend"],
            "blend_minus_lr": diff,
            "ci_low": lo,
            "ci_high": hi,
            "promote": bool(PROMOTE),
        }
    )
    run.finish()

## 4. Findings

From run v1 (2026-10-24, 73 minutes on Kaggle's CPU).

**1. Every Qwen 3 size beats TF-IDF on its own.** The best single layer of each model, on
the seed-7 folds:

| model | best layer | CV |
|---|---|---|
| Qwen 3 0.6B base | 6 of 28 | 0.345 |
| Qwen 3 1.7B base | 3 of 28 | 0.377 |
| Qwen 3 4B base | 8 of 36 | 0.380 |
| TF-IDF `lr` (same folds) | | 0.330 |

**2. Early layers win, and the last layers lose.** The best layers sit at 11–22% of the
depth. The 1.7B and 4B models show a second, lower bump around 55–70%, and the last layers
are the worst of all (0.28–0.32). A plausible reading: early layers still describe the
surface of the text (word choice, punctuation, phrasing), which is much of what tells the
authors apart, while the last layers are specialized for predicting the next token.

**3. Even layer 0 matches TF-IDF.** Layer 0 is the plain average of the input token
embeddings, with no context at all, and it scores 0.33–0.36. Our guess: an average weights
each token by how often it occurs, and frequent tokens (function words, punctuation) carry
style. TF-IDF does the opposite: it boosts rare words, which are mostly about the topic.

**4. Refinement adds a little.** Averaging four layers of the 4B model (6 to 9) with strong
regularization (`C = 0.1`) reaches 0.386.

**5. A small blend weight can hide a big effect.** The best weight on the LM side is 0.1,
and the blend reaches 0.404 on the seed-7 folds, above both of its parts. A weighted mean
mixes *probabilities*, so a very confident model moves the result even at 10%. We suspect
the LM head is much more confident than `lr`; the tuning notebook will check.

**6. The honest A/B on fresh folds (seed 42): promoted.** Blend minus `lr` = **+0.084**,
with a 95% prompt-bootstrap interval of **[+0.066, +0.101]**. The whole interval is above
0. `lr` scores 0.321 on these folds, so the blend scores about 0.405. The table in section
3 also shows seeds 43 and 44.

**Next:** the offline submission notebook extracts the same four layers on Kaggle's GPU and
submits the blend. A public-leaderboard score near 0.40 would confirm that the backtest
still tracks the leaderboard.